# Book Summary Narrator

Pipeline overview:
1. Extract text from a PDF in `sample_data/`
2. Summarize with Sarvam Chat Completions (`sarvam-105b`)
3. Narrate the summary with Text-to-Speech (`bulbul:v3`) into `outputs/`


In [ ]:
%pip install -r requirements.txt


## Setup


In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import PyPDF2
from dotenv import load_dotenv
from sarvamai import SarvamAI
from sarvamai.play import save

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MAX_CHUNK_LENGTH = 500


## Helpers


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    text_parts: list[str] = []
    with pdf_path.open("rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text_parts.append(page.extract_text() or "")
    return "".join(text_parts)


def generate_summary(text: str) -> str:
    prompt = (
        "Please provide a concise summary of the following text. "
        "Focus on the main ideas and key points, keeping the summary "
        "clear and engaging: " + text
    )
    response = client.chat.completions(
        model="sarvam-105b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=2000,
    )
    return response.choices[0].message.content


def split_text_into_chunks(text: str, max_length: int = MAX_CHUNK_LENGTH) -> list[str]:
    sentences = text.replace("\n", " ").split(". ")
    chunks: list[str] = []
    current = ""
    for i, sentence in enumerate(sentences):
        if i != len(sentences) - 1:
            sentence += "."
        if current and len(current) + len(sentence) + 1 > max_length:
            chunks.append(current.strip())
            current = sentence + " "
        else:
            current += sentence + " "
    if current.strip():
        chunks.append(current.strip())
    return chunks


def text_to_speech(text: str, output_path: Path, language_code: str = "en-IN") -> None:
    response = client.text_to_speech.convert(
        text=text,
        target_language_code=language_code,
        speaker="shubh",
        model="bulbul:v3",
    )
    save(response, str(output_path))


## Run


In [ ]:
PDF_PATH = SAMPLE_DIR / "book.pdf"
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PDF_PATH}. Add a text-based PDF under sample_data/ first."
    )

book_text = extract_text_from_pdf(PDF_PATH)
(OUTPUT_DIR / "extracted_text.txt").write_text(book_text, encoding="utf-8")

summary = generate_summary(book_text)
(OUTPUT_DIR / "summary.txt").write_text(summary, encoding="utf-8")
print(summary)

chunks = split_text_into_chunks(summary)
audio_files: list[Path] = []
for i, chunk in enumerate(chunks, start=1):
    audio_path = OUTPUT_DIR / f"summary_narration_part_{i}.wav"
    text_to_speech(chunk, audio_path)
    audio_files.append(audio_path)
    if i < len(chunks):
        time.sleep(1)

print("Audio files:")
for path in audio_files:
    print("-", path)
